# 🫀 Heart Failure Risk Score Prediction
## Aurobindo Subudhi — Threshold Optimization + Final Integration (Updated HRS)

**Responsibilities:** Calibration Curve · Brier Score · Decision Curve Analysis · Threshold Optimization · Final Summary

**Outputs:**
- `Table14_Threshold_Optimization.csv` → Tables/
- `Table15_Final_Summary.csv` → Tables/
- `Plot15_Calibration_Curve.png` → Figures/
- `Plot16_Decision_Curve_Analysis.png` → Figures/
- `Plot18_Threshold_Optimization_Curves.png` → Figures/

### 📁 Folder Setup (Google Colab)
```
/content/
  cleaned_test.csv
  hrs_risk_categories.csv
  lr_model.pkl / LR_coefficients.csv
  rf_model.pkl  |  xgb_model.pkl  |  catboost_model.pkl
```
> ✅ **Fixes applied:** LR included, J-stat used, DCA added, Table 15 added, all files saved.

In [ ]:
# ── 0. Install & Import ────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['xgboost','catboost','scikit-learn','matplotlib',
            'seaborn','pandas','numpy','joblib','scipy']:
    try: __import__(pkg.replace('-','_'))
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

import os, warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
from scipy.special   import expit
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    brier_score_loss, roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble     import RandomForestClassifier
from xgboost  import XGBClassifier
from catboost import CatBoostClassifier

plt.rcParams.update({
    'figure.dpi':150, 'axes.spines.top':False, 'axes.spines.right':False,
    'font.family':'DejaVu Sans', 'axes.titleweight':'bold',
    'axes.titlesize':12, 'axes.labelsize':11, 'legend.fontsize':9,
})
PALETTE = ['#2C7BB6','#D7191C','#1A9641','#FDAE61','#7B2D8B']
print('✅ All libraries imported.')

In [ ]:
# ── 1. Set Paths ───────────────────────────────────────────────────────────
OUTPUTS_DIR = '/content'      # Google Colab
FIGURES_DIR = '/content/Figures'
TABLES_DIR  = '/content/Tables'

# ── Uncomment for local use ────────────────────────────────────────────────
# OUTPUTS_DIR = '.'
# FIGURES_DIR = './Figures'
# TABLES_DIR  = './Tables'

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR,  exist_ok=True)
print('✅ Paths set.')

In [ ]:
# ── 2. Load Test Data ──────────────────────────────────────────────────────
test_df = pd.read_csv(os.path.join(OUTPUTS_DIR, 'cleaned_test.csv'))
TARGET  = 'target'

if TARGET not in test_df.columns:
    binary_cols = [c for c in test_df.columns
                   if test_df[c].nunique()==2 and test_df[c].dtype in ['int64','float64']]
    TARGET = binary_cols[-1]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET].astype(int)

print(f'Features  : {X_test.shape[1]}')
print(f'Samples   : {len(y_test)}')
print(f'Positives : {y_test.sum()} ({y_test.mean()*100:.1f}%)')

In [ ]:
# ── 3. Load All Models (including LR) ──────────────────────────────────────
def load_pkl(fname):
    path = os.path.join(OUTPUTS_DIR, fname)
    if not os.path.exists(path): return None
    return joblib.load(path)

print('Loading models...')

lr_model = load_pkl('lr_model.pkl')
if lr_model is None:
    print('  ⚠️ lr_model.pkl not found — building from LR_coefficients.csv')
    lr_coefs  = pd.read_csv(os.path.join(OUTPUTS_DIR, 'LR_coefficients.csv'))
    int_s     = lr_coefs[lr_coefs['Feature']=='intercept']['Coefficient']
    intercept = 0.0 if int_s.empty else float(int_s.iloc[0])
    coef_map  = lr_coefs[lr_coefs['Feature']!='intercept'].set_index('Feature')['Coefficient'].to_dict()
    class LRWrapper:
        def __init__(self,c,b): self.c=c; self.b=b
        def _s(self,X):
            s=self.b
            for f,w in self.c.items():
                if f in X.columns: s=s+X[f].values*w
            return s
        def predict_proba(self,X):
            p=1/(1+np.exp(-self._s(X))); return np.column_stack([1-p,p])
        def predict(self,X): return (self.predict_proba(X)[:,1]>=0.5).astype(int)
    lr_model = LRWrapper(coef_map, intercept)

rf_model  = load_pkl('rf_model.pkl')  or (lambda: (print('  ⚠️ RF fallback'), RandomForestClassifier(100,random_state=42).fit(X_test,y_test))[1])()
xgb_model = load_pkl('xgb_model.pkl') or (lambda: (print('  ⚠️ XGB fallback'), XGBClassifier(100,random_state=42,eval_metric='logloss',verbosity=0).fit(X_test,y_test))[1])()
cat_model = load_pkl('catboost_model.pkl')
if cat_model is None:
    cbm_path = os.path.join(OUTPUTS_DIR,'catboost_model.cbm')
    if os.path.exists(cbm_path): cat_model=CatBoostClassifier(); cat_model.load_model(cbm_path)
    else:
        print('  ⚠️ CatBoost fallback')
        cat_model=CatBoostClassifier(100,random_seed=42,verbose=0); cat_model.fit(X_test,y_test)

models = {'LR':lr_model,'RF':rf_model,'XGBoost':xgb_model,'CatBoost':cat_model}
print('\n✅ All models loaded.')

In [ ]:
# ── 4. Load Updated HRS + Get All Probabilities ────────────────────────────
hrs_df  = pd.read_csv(os.path.join(OUTPUTS_DIR, 'hrs_risk_categories.csv'))
hrs_raw = hrs_df['HRS_score'].values

# ✅ IMPROVED: Sigmoid normalization
prob_hrs = expit((hrs_raw - hrs_raw.mean()) / hrs_raw.std())
pred_hrs = (hrs_df['risk_class']=='High').astype(int).values

# Collect all model probabilities
prob_lr  = lr_model.predict_proba(X_test)[:,1]
prob_rf  = rf_model.predict_proba(X_test)[:,1]
prob_xgb = xgb_model.predict_proba(X_test)[:,1]
prob_cat = cat_model.predict_proba(X_test)[:,1]

MODEL_NAMES = ['LR','RF','XGBoost','CatBoost','HRS']
PROBS       = [prob_lr, prob_rf, prob_xgb, prob_cat, prob_hrs]

# Brier scores
BRIER = {n: round(brier_score_loss(y_test, p),4) for n,p in zip(MODEL_NAMES,PROBS)}
print('Brier Scores (lower = better | 0.25 = no skill):')
for n,b in BRIER.items():
    bar = '█'*int(b*40)
    print(f'  {n:10s}: {b:.4f}  {bar}')
print('\n✅ HRS loaded with sigmoid normalization.')

In [ ]:
# ── 5. Plot 15 — Calibration Curves ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.plot([0,1],[0,1],'k--', lw=1.5, label='Perfect calibration')

for name,p,color in zip(MODEL_NAMES,PROBS,PALETTE):
    fp,mp = calibration_curve(y_test, p, n_bins=10, strategy='uniform')
    ax.plot(mp, fp, marker='o', markersize=5, lw=2.2, color=color,
            label=f'{name}  (Brier={BRIER[name]:.4f})')

ax.set_xlabel('Mean Predicted Probability', fontsize=11)
ax.set_ylabel('Fraction of Positives',      fontsize=11)
ax.set_title('Plot 15: Calibration Curves — All Models\nHeart Failure Risk Score Prediction',
             fontsize=13, pad=12)
ax.legend(loc='upper left', framealpha=0.9)
ax.set_xlim(-0.02,1.02); ax.set_ylim(-0.02,1.10)
ax.grid(axis='both', linestyle='--', alpha=0.25)
ax.annotate('Closer to dashed line = better calibration',
            xy=(0.5,0.03), fontsize=8, color='gray', ha='center', style='italic')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR,'Plot15_Calibration_Curve.png'),
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('✅ Plot 15 saved.')

In [ ]:
# ── 6. Plot 16 — Decision Curve Analysis (DCA) ─────────────────────────────
def net_benefit(y_true, prob, t):
    pred = (prob >= t).astype(int)
    n  = len(y_true)
    tp = ((pred==1)&(y_true==1)).sum()
    fp = ((pred==1)&(y_true==0)).sum()
    return (tp/n) - (t/(1-t+1e-9))*(fp/n)

thresholds = np.linspace(0.01, 0.99, 99)
prevalence = y_test.mean()
nb_all  = [prevalence-(t/(1-t+1e-9))*(1-prevalence) for t in thresholds]
nb_none = [0.0]*len(thresholds)

fig, ax = plt.subplots(figsize=(9, 6.5))
ax.plot(thresholds, nb_all,  color='#888', lw=1.8, ls='--', label='Treat All')
ax.plot(thresholds, nb_none, color='#333', lw=1.8, ls=':',  label='Treat None')
for name,p,color in zip(MODEL_NAMES,PROBS,PALETTE):
    nb = [net_benefit(y_test.values, p, t) for t in thresholds]
    ax.plot(thresholds, nb, lw=2.3, color=color, label=name)
ax.axhline(0, color='k', lw=0.7, alpha=0.4)
ax.set_xlabel('Threshold Probability', fontsize=11)
ax.set_ylabel('Net Benefit',           fontsize=11)
ax.set_title('Plot 16: Decision Curve Analysis\nHeart Failure Risk Score Prediction',
             fontsize=13, pad=12)
ax.set_xlim(0,1); ax.set_ylim(-0.06, prevalence+0.08)
ax.legend(loc='upper right', framealpha=0.9)
ax.grid(axis='both', linestyle='--', alpha=0.25)
ax.annotate('Higher net benefit = better clinical utility',
            xy=(0.5,-0.055), fontsize=8, color='gray', ha='center', style='italic')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR,'Plot16_Decision_Curve_Analysis.png'),
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('✅ Plot 16 saved.')

In [ ]:
# ── 7. Threshold Optimization (Youden J-stat) ──────────────────────────────
def sweep(y_true, prob):
    rows = []
    for t in np.arange(0.05, 0.96, 0.05):
        pred = (prob >= t).astype(int)
        prec = precision_score(y_true, pred, zero_division=0)
        rec  = recall_score(y_true, pred, zero_division=0)
        f1   = f1_score(y_true, pred, zero_division=0)
        spec = (((pred==0)&(y_true==0)).sum() / max((y_true==0).sum(),1))
        j    = rec + spec - 1
        rows.append({'Threshold':round(t,2),'Precision':prec,'Recall':rec,'F1':f1,'J_stat':j})
    return pd.DataFrame(rows)

sweep_results = {}
opt_thresh    = {}

for name,p in zip(MODEL_NAMES,PROBS):
    df = sweep(y_test.values, p)
    sweep_results[name] = df
    best = df.loc[df['J_stat'].idxmax()]
    opt_thresh[name] = best['Threshold']
    print(f'{name:10s} → optimal threshold={best["Threshold"]:.2f}  '
          f'J={best["J_stat"]:.3f}  F1={best["F1"]:.3f}')

In [ ]:
# ── 8. Table 14 — Threshold Optimization Table ─────────────────────────────
rows14 = []
for name,p in zip(MODEL_NAMES,PROBS):
    best = sweep_results[name].loc[sweep_results[name]['J_stat'].idxmax()]
    rows14.append({
        'Model'            : name,
        'Optimal Threshold': best['Threshold'],
        'Precision'        : round(best['Precision'],4),
        'Recall'           : round(best['Recall'],4),
        'F1 Score'         : round(best['F1'],4),
        'Youden J-stat'    : round(best['J_stat'],4),
    })

table14 = pd.DataFrame(rows14)
table14.to_csv(os.path.join(TABLES_DIR,'Table14_Threshold_Optimization.csv'), index=False)

print('📋 Table 14: Threshold Optimization Table')
display(table14.style
    .background_gradient(subset=['Youden J-stat','F1 Score'], cmap='YlGn')
    .format('{:.4f}', subset=['Optimal Threshold','Precision','Recall','F1 Score','Youden J-stat'])
    .set_caption('Table 14: Optimal Thresholds per Model (Youden J-stat)'))
print('\n✅ Table 14 saved.')

In [ ]:
# ── 9. Plot 18 — Threshold Optimization Curves ─────────────────────────────
mc = {'Precision':'#2C7BB6','Recall':'#D7191C','F1':'#1A9641','J_stat':'#FDAE61'}
ml = {'Precision':'Precision','Recall':'Recall','F1':'F1 Score','J_stat':'Youden J-stat'}

fig = plt.figure(figsize=(17,10))
gs  = gridspec.GridSpec(2,3,figure=fig,hspace=0.42,wspace=0.32)

for idx,(name,p,color) in enumerate(zip(MODEL_NAMES,PROBS,PALETTE)):
    row,col = divmod(idx,3)
    ax = fig.add_subplot(gs[row,col])
    df = sweep_results[name]
    opt_t = opt_thresh[name]
    for m in ['Precision','Recall','F1','J_stat']:
        ax.plot(df['Threshold'],df[m],lw=2.0,color=mc[m],label=ml[m])
    ax.axvline(opt_t,color='black',lw=1.8,ls='--',alpha=0.85,
               label=f'Optimal: {opt_t:.2f}')
    j_val = df.loc[df['J_stat'].idxmax(),'J_stat']
    ax.scatter([opt_t],[j_val],color='black',s=60,zorder=5)
    ax.set_title(f'{name}   (Best J={j_val:.3f})',fontsize=11,fontweight='bold',color=color,pad=8)
    ax.set_xlabel('Threshold',fontsize=9); ax.set_ylabel('Score',fontsize=9)
    ax.set_xlim(0,1); ax.set_ylim(-0.05,1.08)
    ax.legend(fontsize=7.5,loc='lower right',framealpha=0.88)
    ax.grid(axis='both',linestyle='--',alpha=0.25)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

fig.add_subplot(gs[1,2]).set_visible(False)
fig.suptitle('Plot 18: Threshold Optimization Curves — All Models\nHeart Failure Risk Score Prediction',
             fontsize=14,fontweight='bold',y=1.01)
plt.savefig(os.path.join(FIGURES_DIR,'Plot18_Threshold_Optimization_Curves.png'),
            dpi=150,bbox_inches='tight',facecolor='white')
plt.show()
print('✅ Plot 18 saved.')

In [ ]:
# ── 10. Table 15 — Final Summary Table ─────────────────────────────────────
rows15 = []
for name,p in zip(MODEL_NAMES,PROBS):
    opt_t = opt_thresh[name]
    pred  = (p >= opt_t).astype(int)
    tn,fp,fn,tp = confusion_matrix(y_test,pred,labels=[0,1]).ravel()
    sens = tp/max(tp+fn,1); spec = tn/max(tn+fp,1)
    rows15.append({
        'Model'        : name,
        'Threshold'    : opt_t,
        'AUC-ROC'      : round(roc_auc_score(y_test,p),4),
        'Brier Score'  : BRIER[name],
        'Accuracy'     : round(accuracy_score(y_test,pred),4),
        'Sensitivity'  : round(sens,4),
        'Specificity'  : round(spec,4),
        'Precision'    : round(precision_score(y_test,pred,zero_division=0),4),
        'F1 Score'     : round(f1_score(y_test,pred,zero_division=0),4),
        'Youden J-stat': round(sens+spec-1,4),
    })

table15 = pd.DataFrame(rows15)
table15['Clinical Score'] = (
    0.35*table15['AUC-ROC'] + 0.25*table15['F1 Score'] +
    0.20*table15['Sensitivity'] + 0.20*(1-table15['Brier Score'])
).round(4)
table15 = table15.sort_values('Clinical Score',ascending=False).reset_index(drop=True)
table15.to_csv(os.path.join(TABLES_DIR,'Table15_Final_Summary.csv'),index=False)

print('📋 Table 15: Final Summary Table')
display(table15.style
    .background_gradient(subset=['AUC-ROC','F1 Score','Clinical Score'], cmap='YlGn')
    .background_gradient(subset=['Brier Score'], cmap='YlOrRd_r')
    .format('{:.4f}')
    .set_caption('Table 15: Final Model Summary at Optimal Thresholds'))
print('\n✅ Table 15 saved.')

In [ ]:
# ── 11. Best Model & Final Summary ─────────────────────────────────────────
best = table15.iloc[0]
print('='*55)
print(f'  🏆 BEST MODEL: {best["Model"]}')
print('='*55)
for col in ['AUC-ROC','Brier Score','Accuracy','Sensitivity',
            'Specificity','F1 Score','Youden J-stat','Clinical Score']:
    print(f'  {col:20s}: {best[col]:.4f}')
print(f'  Optimal Threshold    : {best["Threshold"]:.2f}')
print('='*55)
print()
print('All outputs saved:')
for folder in ['Tables','Figures']:
    fpath = TABLES_DIR if folder=='Tables' else FIGURES_DIR
    for f in sorted(os.listdir(fpath)):
        size = os.path.getsize(os.path.join(fpath,f))
        print(f'  ✅ {folder}/{f}  ({size:,} bytes)')
print()
print('📌 Upload Tables/ and Figures/ to GitHub → raise Pull Request')